In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
%cd /content/drive/MyDrive/MSIB/

/content/drive/MyDrive/MSIB


## Topic Modeling - Nouns only

In [32]:
import nltk
from nltk import word_tokenize, pos_tag
nltk.download('averaged_perceptron_tagger')

def nouns(text):
  is_noun = lambda pos: pos[:2] == 'NN'
  tokenized = word_tokenize(text)
  all_nouns = [word for (word, pos) in pos_tag(tokenized) if is_noun(pos)]
  return ' '.join(all_nouns)

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [33]:
import pandas as pd
data_clean = pd.read_pickle('corpus.pkl')
data_clean

,transcript,full_name
blackwidow,Following the events of Captain America: Civ...,Spider-man: No Way Home
eternals,"The saga of the Eternals, a race of immortal...",Eternals
joker,CLASSIC SCENE Arthur is invited to appear on...,Wonder Woman
justiceleague,Determined to ensure Superman’s ultimate sac...,Joker
shangchi,Shang-Chi is a young man who is in denial ab...,Justice League (Jack Snyder)
spiderman,Peter Parker’s secret identity is revealed t...,Shang Chi and the Legend of the Ten Rings
venom,"In 1996, a young Cletus Kasady watches helpl...",Venom: Let There be Carnage
wonderwoman,"As a young girl, Diana Prince participates i...",Black Widow


In [31]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [34]:
data_nouns = pd.DataFrame(data_clean.transcript.apply(nouns))
data_nouns

,transcript
blackwidow,events Captain America Civil War Natasha Roman...
eternals,saga Eternals race beings Earth history civili...
joker,CLASSIC SCENE Arthur Murray ’ s show popularit...
justiceleague,Superman ’ s sacrifice vain Bruce Wayne forces...
shangchi,Shang-Chi man vocation magnificent warrior des...
spiderman,Peter Parker ’ identity world Desperate help P...
venom,Cletus Kasady love Frances Barrison St. Estes ...
wonderwoman,girl Diana Prince competition Themyscira Amazo...


In [35]:
from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import CountVectorizer

add_stop_words = ['like', 'im', 'know', 'just', 'dont', 'thats', 'right', 'people',
                  'youre', 'got', 'gonna', 'time', 'think', 'yeah', 'said']
stop_words = list(text.ENGLISH_STOP_WORDS.union(add_stop_words))

cvn = CountVectorizer(stop_words=stop_words)
data_cvn = cvn.fit_transform(data_nouns.transcript)
data_dtmn = pd.DataFrame(data_cvn.toarray(), columns=cvn.get_feature_names_out())
data_dtmn.index = data_nouns.index
data_dtmn

,aah,abductions,abilities,ability,aboard,abode,abraham,abrasion,absence,absolute,...,youth,yup,zero,zeroes,zeus,zhong,zip,zones,zoologist,zoom
blackwidow,0,0,0,0,0,1,0,0,1,0,...,0,1,1,0,0,0,1,1,0,0
eternals,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
joker,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
justiceleague,0,2,2,0,1,0,0,1,1,1,...,0,0,0,1,2,0,0,0,0,1
shangchi,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
spiderman,0,0,0,1,0,0,0,0,0,0,...,1,2,0,0,0,0,0,0,0,0
venom,1,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
wonderwoman,0,0,0,0,1,0,1,0,0,0,...,1,0,0,0,0,2,0,0,1,0


In [36]:
!pip install scipy


[notice] A new release of pip is available: 23.0 -> 23.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
!pip install gensim


[notice] A new release of pip is available: 23.0 -> 23.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
import gensim
import scipy
from gensim import models
from gensim import interfaces, utils, matutils
from gensim.matutils import dirichlet_expectation, mean_absolute_difference

In [39]:
# create gensim corpus
corpusn = matutils.Sparse2Corpus(scipy.sparse.csr_matrix(data_dtmn.transpose()))

# create vocab dict
id2wordn = dict((v,k) for k, v in cvn.vocabulary_.items())

In [24]:
# start with 2 topics
ldan = models.LdaModel(corpus=corpusn, num_topics=2, id2word=id2wordn, passes=10)
ldan.print_topics()

[(0,
  '0.013*"okay" + 0.012*"peter" + 0.010*"man" + 0.010*"hey" + 0.008*"diana" + 0.007*"music" + 0.007*"grunts" + 0.006*"playing" + 0.006*"grunting" + 0.006*"oh"'),
 (1,
  '0.013*"music" + 0.012*"grunts" + 0.010*"playing" + 0.008*"sersi" + 0.008*"okay" + 0.007*"bruce" + 0.006*"man" + 0.006*"barry" + 0.006*"world" + 0.005*"mother"')]

In [40]:
# with 3 topics
ldan = models.LdaModel(corpus=corpusn, num_topics=3, id2word=id2wordn, passes=10)
ldan.print_topics()

[(0,
  '0.019*"okay" + 0.016*"peter" + 0.010*"man" + 0.010*"hey" + 0.008*"gon" + 0.008*"eddie" + 0.007*"don" + 0.007*"natasha" + 0.007*"spider" + 0.006*"sorry"'),
 (1,
  '0.020*"music" + 0.016*"grunting" + 0.016*"playing" + 0.013*"grunts" + 0.012*"katy" + 0.010*"mandarin" + 0.009*"morris" + 0.009*"english" + 0.009*"shaun" + 0.008*"shang"'),
 (2,
  '0.014*"diana" + 0.014*"music" + 0.011*"grunts" + 0.011*"playing" + 0.009*"sersi" + 0.008*"world" + 0.008*"man" + 0.007*"bruce" + 0.006*"okay" + 0.006*"barry"')]

In [41]:
# with 4 topics
ldan = models.LdaModel(corpus=corpusn, num_topics=4, id2word=id2wordn, passes=10)
ldan.print_topics()

[(0,
  '0.014*"sersi" + 0.014*"diana" + 0.013*"playing" + 0.013*"music" + 0.009*"barbara" + 0.008*"grunts" + 0.008*"world" + 0.007*"steve" + 0.007*"maxwell" + 0.007*"ikaris"'),
 (1,
  '0.018*"music" + 0.016*"grunts" + 0.012*"playing" + 0.011*"bruce" + 0.010*"grunting" + 0.009*"barry" + 0.008*"mother" + 0.008*"victor" + 0.008*"man" + 0.007*"diana"'),
 (2,
  '0.024*"eddie" + 0.015*"brock" + 0.013*"kasady" + 0.012*"venom" + 0.010*"hey" + 0.009*"okay" + 0.009*"cletus" + 0.008*"man" + 0.006*"anne" + 0.005*"uh"'),
 (3,
  '0.022*"peter" + 0.021*"okay" + 0.011*"man" + 0.009*"gon" + 0.009*"natasha" + 0.009*"hey" + 0.009*"spider" + 0.008*"don" + 0.008*"melina" + 0.007*"yelena"')]

## Topic Modeling - Nouns and Adj

In [27]:
def nouns_adj(text):
  is_noun_adj = lambda pos: pos[:2] == 'NN' or pos[:2] == 'JJ'
  tokenized = word_tokenize(text)
  nouns_adj = [word for (word, pos) in pos_tag(tokenized) if is_noun_adj(pos)]
  return ' '.join(nouns_adj)

In [42]:
data_nouns_adj = pd.DataFrame(data_clean.transcript.apply(nouns_adj))
data_nouns_adj

,transcript
blackwidow,events Captain America Civil War Natasha Roman...
eternals,saga Eternals race immortal beings Earth histo...
joker,CLASSIC SCENE Arthur Murray ’ s show due popul...
justiceleague,Superman ’ s ultimate sacrifice vain Bruce Way...
shangchi,Shang-Chi young man denial vocation magnificen...
spiderman,Peter Parker ’ secret identity entire world De...
venom,young Cletus Kasady love Frances Barrison St. ...
wonderwoman,young girl Diana Prince multi-stage athletic c...


In [44]:
# create new document-term matrix
cvna = CountVectorizer(stop_words=stop_words, max_df=.8)
data_cvna = cvna.fit_transform(data_nouns_adj.transcript)
data_dtmna = pd.DataFrame(data_cvna.toarray(), columns=cvna.get_feature_names_out())
data_dtmna.index = data_nouns.index
data_dtmna

,20,300,aah,abductions,abilities,ability,aboard,abode,abraham,abrasion,...,youth,yup,zero,zeroes,zeus,zhong,zip,zones,zoologist,zoom
blackwidow,1,0,0,0,0,0,0,1,0,0,...,0,1,1,0,0,0,1,1,0,0
eternals,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
joker,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
justiceleague,0,0,0,2,2,0,1,0,0,1,...,0,0,0,1,2,0,0,0,0,1
shangchi,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
spiderman,0,0,0,0,0,1,0,0,0,0,...,1,2,0,0,0,0,0,0,0,0
venom,0,1,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
wonderwoman,0,0,0,0,0,0,1,0,1,0,...,1,0,0,0,0,2,0,0,1,0


In [45]:
# create gensim corpus
corpusna = matutils.Sparse2Corpus(scipy.sparse.csr_matrix(data_dtmna.transpose()))

# create vocab dict
id2wordna = dict((v,k) for k, v in cvna.vocabulary_.items())

In [46]:
# Let's start with 2 topics
ldana = models.LdaModel(corpus=corpusna, num_topics=2, id2word=id2wordna, passes=10)
ldana.print_topics()

[(0,
  '0.017*"music" + 0.013*"playing" + 0.012*"grunts" + 0.011*"sersi" + 0.009*"bruce" + 0.008*"barry" + 0.007*"victor" + 0.006*"diana" + 0.006*"box" + 0.006*"thena"'),
 (1,
  '0.010*"peter" + 0.009*"grunts" + 0.008*"grunting" + 0.007*"diana" + 0.007*"music" + 0.006*"playing" + 0.006*"english" + 0.005*"spider" + 0.005*"barbara" + 0.005*"eddie"')]

In [47]:
# Let's try 3 topics
ldana = models.LdaModel(corpus=corpusna, num_topics=3, id2word=id2wordna, passes=10)
ldana.print_topics()

[(0,
  '0.014*"sersi" + 0.014*"diana" + 0.014*"playing" + 0.014*"music" + 0.010*"barbara" + 0.008*"grunts" + 0.008*"steve" + 0.008*"maxwell" + 0.007*"ikaris" + 0.007*"thena"'),
 (1,
  '0.016*"grunting" + 0.014*"grunts" + 0.013*"music" + 0.011*"playing" + 0.010*"english" + 0.009*"shang" + 0.008*"natasha" + 0.008*"chi" + 0.008*"katy" + 0.007*"melina"'),
 (2,
  '0.015*"peter" + 0.009*"bruce" + 0.008*"grunts" + 0.008*"barry" + 0.008*"spider" + 0.007*"eddie" + 0.007*"victor" + 0.006*"music" + 0.006*"box" + 0.006*"diana"')]

In [48]:
# Let's try 4 topics
ldana = models.LdaModel(corpus=corpusna, num_topics=4, id2word=id2wordna, passes=10)
ldana.print_topics()

[(0,
  '0.019*"diana" + 0.013*"barbara" + 0.013*"eddie" + 0.011*"steve" + 0.010*"maxwell" + 0.009*"wish" + 0.008*"brock" + 0.007*"kasady" + 0.007*"venom" + 0.005*"alistair"'),
 (1,
  '0.039*"peter" + 0.019*"spider" + 0.012*"parker" + 0.008*"wait" + 0.008*"spell" + 0.007*"murray" + 0.007*"ned" + 0.007*"mj" + 0.005*"joker" + 0.005*"sir"'),
 (2,
  '0.017*"bruce" + 0.015*"grunts" + 0.015*"barry" + 0.013*"victor" + 0.012*"music" + 0.011*"diana" + 0.011*"box" + 0.008*"alfred" + 0.008*"steppenwolf" + 0.007*"silas"'),
 (3,
  '0.019*"music" + 0.018*"playing" + 0.014*"grunts" + 0.013*"grunting" + 0.011*"sersi" + 0.009*"english" + 0.006*"continues" + 0.006*"shang" + 0.006*"natasha" + 0.006*"chi"')]

## Indetify Topics in Each Document

In [49]:
ldana = models.LdaModel(corpus=corpusna, num_topics=4, id2word=id2wordna, passes=80)
ldana.print_topics()

[(0,
  '0.042*"peter" + 0.020*"spider" + 0.013*"parker" + 0.009*"wait" + 0.008*"spell" + 0.007*"ned" + 0.007*"mj" + 0.005*"sir" + 0.004*"dude" + 0.004*"strange"'),
 (1,
  '0.015*"sersi" + 0.014*"diana" + 0.014*"playing" + 0.014*"music" + 0.010*"barbara" + 0.008*"grunts" + 0.008*"steve" + 0.008*"maxwell" + 0.008*"ikaris" + 0.008*"thena"'),
 (2,
  '0.017*"grunts" + 0.015*"music" + 0.011*"grunting" + 0.010*"playing" + 0.008*"bruce" + 0.007*"english" + 0.007*"barry" + 0.006*"victor" + 0.006*"shang" + 0.006*"natasha"'),
 (3,
  '0.025*"eddie" + 0.015*"brock" + 0.013*"kasady" + 0.013*"venom" + 0.009*"cletus" + 0.007*"anne" + 0.005*"dan" + 0.005*"barrison" + 0.004*"mm" + 0.004*"mulligan"')]

In [39]:
corpus_transformed = ldana[corpusna]
list(zip([a for [(a,b)] in corpus_transformed], data_dtmna.index))

[(3, 'blackwidow'),
 (1, 'eternals'),
 (0, 'joker'),
 (3, 'justiceleague'),
 (1, 'shangchi'),
 (1, 'spiderman'),
 (0, 'venom'),
 (3, 'wonderwoman')]